# Take-Home Assignment: Managing CO2 Emissions Data with PyIceberg

**Objective:** build and manage an Iceberg lakehouse using global CO2 emissions
data. You will create tables, evolve a schema, time travel through table history,
partition for performance, and perform an atomic upsert.

**Dataset:** Our World in Data (OWID) Global CO2 Emissions.

### How this relates to the in-class demo

The demo notebook walked through every one of these operations on World Bank GDP
data, with the reasoning spelled out in the ❄️ callouts. This assignment asks you
to perform the same operations on a different dataset, writing the Iceberg code
yourself.

The data cleaning is **not** what is being assessed, so it is provided for you
through `co2_pipeline.py`, exactly as the demo used `gdp_pipeline.py`. Both modules
re-export the shared helpers from `iceberg_helpers.py`:

| Helper | What it does |
|---|---|
| `load_co2_csv(path)` | read a source CSV and print a summary |
| `clean_co2(df, impute=...)` | filter aggregate rows and impute missing `co2` values |
| `infer_iceberg_schema(df)` | build a typed Iceberg `Schema` from a DataFrame |
| `compare_schemas(table, df)` | return `(dropped, added)` columns and print a report |
| `show_warehouse(table)` | list the physical files backing a table |
| `find_snapshot(table, contains=...)` | look a snapshot up by its commit comment |
| `reset_tutorial(catalog, namespaces=...)` | drop everything and start over |

Read `co2_pipeline.py` if you want to know exactly what the cleaning does.

### Variable names matter

Later cells depend on earlier ones. Where a cell says *name it `X`*, use that
exact name or the rest of the notebook will not run.

## Grading Rubric (100% total)

| Section & Weight | Excellent (full section weight) | Partial Credit (50%) | Incomplete (0%) |
| :--- | :--- | :--- | :--- |
| **Part 1** *(10%)* | Correctly initializes the REST catalog and prints the list of namespaces. | Catalog is initialized, but the URI is incorrect, or the namespace list fails to print. | Missing, or code fails to execute. |
| **Part 2** *(10%)* | Builds the schema programmatically by iterating over dtypes and using the type map. Creates the namespace and table. Converts to PyArrow and appends with all three snapshot properties. | Schema is hardcoded rather than generated, or `infer_iceberg_schema()` is called instead of implementing the loop. Data appended but snapshot properties missing. | Schema construction fails, or data is not appended. |
| **Part 3** *(10%)* | Creates the processed namespace and table. Converts the cleaned DataFrame to PyArrow and appends it. | Table created but data appended as a Pandas DataFrame without PyArrow conversion, or minor syntax errors prevent execution. | Namespace/table creation fails, or data is not appended. |
| **Part 4** *(10%)* | Identifies dropped columns programmatically. Drops them inside the `update_schema()` context manager. Overwrites the table with the 2021 data and records the change in snapshot properties. | Identifies dropped columns by hand, drops them without the context manager, or appends instead of overwriting. | Schema is not updated, or the overwrite fails. |
| **Part 5** *(10%)* | Lists all snapshots with operation and comment. Retrieves the 2020 state using `find_snapshot()` rather than a positional index, and shows both the row count and the restored `source` column. | Retrieves a previous snapshot but by hard-coded index or ID, or fails to show that the dropped column returns. | Time travel is not attempted, or throws a fatal error. |
| **Part 6** *(20%)* | Looks the partition field ID up by name, builds a correct `PartitionSpec`, creates and populates the partitioned table, and demonstrates pruning by comparing planned files and records. | Table created without the partition spec, partition field ID hard-coded, or the pruning comparison is missing/incorrect. | Fails to create the partitioned table. |
| **Part 7** *(20%)* | Stages both the delete and the append inside a single `transaction()` block, commits once, and verifies against the **raw** table that the five countries now have 2021 rows. | Operations executed outside the transaction, or as two separate commits. Verification missing or filtered on the wrong table/countries. | Upsert not attempted, or transaction fails to commit. |
| **Part 8** *(10%)* | Answers all four questions in their own words, demonstrating conceptual understanding of schema evolution, time travel, partitioning, and the metadata layer, citing evidence from the notebook output. | Answers are brief or merely restate what the code did without conceptual understanding. | Missing, or does not address the questions. |

## Part 1: Setup and connecting to the catalog

Import the libraries and connect to your local Iceberg REST Catalog.

In [ ]:
import pandas as pd
import pyarrow as pa

from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, DoubleType, LongType, BooleanType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.expressions import And, In, EqualTo
from pyiceberg.exceptions import NoSuchTableError, NamespaceAlreadyExistsError

# Data-processing helpers. Everything in here is plain pandas - see co2_pipeline.py.
from co2_pipeline import (
    load_co2_csv,
    clean_co2,
    infer_iceberg_schema,
    compare_schemas,
    show_warehouse,
    find_snapshot,
    reset_tutorial,
)

DATA_DIR = "/tmp/warehouse"
NAMESPACES = ("co2.raw_data", "co2.processed_data")

---
### ❄️ Iceberg Operation — CONNECT TO THE CATALOG

**What you are doing:** opening a connection to the Iceberg REST catalog at `http://iceberg-rest-catalog:8181`. Name the connection `catalog`.

**Watch for in the output:** an empty list of namespaces on a clean run.

**Corresponds to:** Figure 1 in the paper.

In [ ]:
# TODO: initialize the catalog. Name it `catalog`.
# Hint: load_catalog() takes a name and a dict of properties containing "uri".

catalog =  # YOUR CODE HERE

print("Namespaces:", catalog.list_namespaces())

> **Lost your place?** Uncomment and run the cell below to drop every table and
> namespace this assignment creates, then start again from the top. Snapshot counts
> only make sense on a clean run — if you re-run a write cell, you add a snapshot.

In [ ]:
# reset_tutorial(catalog, namespaces=NAMESPACES)

## Part 2: Loading the raw data (up to 2020)

Load the historical CO2 data up to 2020, build a schema for it, and write it to a
new Iceberg table.

*Data preparation step — ordinary pandas work, no Iceberg involved. Provided for you.*

Read the source CSV. No transformation is applied — this is the raw dataset.

In [ ]:
df_2020 = load_co2_csv(f"{DATA_DIR}/co2_data_until_2020.csv")

df_2020.head()

---
### ❄️ Iceberg Operation — CREATE TABLE

**What you are doing:** building a typed Iceberg schema from the DataFrame's dtypes,
then creating an empty table with it. Name the schema `inferred_iceberg_schema` and
the table `iceberg_raw_table`.

**Note:** the demo called `infer_iceberg_schema(df)` to do this in one line. Here you
implement that helper's logic yourself, because the mapping from pandas dtypes to
Iceberg types — and the fact that every field carries a stable numeric ID — is the
thing being assessed. You may call `infer_iceberg_schema()` afterwards to check your
answer, but the graded cell must be your own loop.

**Watch for in the output:** a `field_id` on every column, numbered from 1.

In [ ]:
# Mapping Pandas dtypes to PyIceberg types
pandas_to_iceberg_typemap = {
    'object': StringType(),
    'int64': LongType(),
    'float64': DoubleType(),
    'bool': BooleanType(),
}

# TODO: iterate over df_2020.dtypes and build a list of NestedField objects.
#   - field_id must start at 1 and increment
#   - name is the column name
#   - field_type comes from the type map (default to StringType() if not found)
#   - required=False
# Then build a Schema from that list and name it `inferred_iceberg_schema`.

iceberg_fields = []
# YOUR CODE HERE

inferred_iceberg_schema =  # YOUR CODE HERE

print(inferred_iceberg_schema)

In [ ]:
namespace_name = "co2.raw_data"
table_name = f"{namespace_name}.emissions"

# TODO: create the namespace if it does not exist (catch NamespaceAlreadyExistsError).
# TODO: drop the table if it already exists (catch NoSuchTableError).
# TODO: create the empty Iceberg table with your inferred schema.
#       Name the resulting table object `iceberg_raw_table`.

# YOUR CODE HERE

---
### ❄️ Iceberg Operation — APPEND

**What you are doing:** converting `df_2020` to a PyArrow table and appending it, tagged with snapshot properties. Include a `job-name`, a `source-file`, and a `comment`.

**Watch for in the output:** a snapshot with operation `append`. Write a *descriptive* comment — you will use it in Part 5 to find this exact version again.

**Corresponds to:** Figure 2 in the paper.

In [ ]:
# TODO: convert df_2020 to a PyArrow table and append it to iceberg_raw_table.
#       Pass snapshot_properties with 'job-name', 'source-file' and 'comment'.
#       Make the comment mention 2020 - Part 5 depends on being able to find it.

# YOUR CODE HERE

for snapshot in iceberg_raw_table.snapshots():
    print(snapshot.snapshot_id, "|", snapshot.summary.get("operation"), "|",
          snapshot.summary.get("comment"))

Provided for you: a look at what Iceberg actually wrote to disk. Keep an eye on
these numbers — Part 4 and Part 8 both come back to them.

In [ ]:
show_warehouse(iceberg_raw_table, label="after the first append")

## Part 3: Cleaning and the processed table

The raw data contains gaps, and OWID mixes continents and income groups in with
individual countries. We clean it and store the result in a *separate* table, so
that the provider's history and our own analytical history stay independent.

*Data preparation step — ordinary pandas work, no Iceberg involved. Provided for you.*

`clean_co2()` drops aggregate rows, drops countries with no data at all, and imputes the remaining gaps in `co2` by forward then backward fill within each country. See `co2_pipeline.py` for the details.

In [ ]:
df_raw = iceberg_raw_table.scan().to_pandas()

df_processed = clean_co2(df_raw, impute="ffill")

df_processed.head()

---
### ❄️ Iceberg Operation — CREATE TABLE + APPEND

**What you are doing:** creating a second table in the `co2.processed_data` namespace and appending the cleaned data. Name the table object `iceberg_processed_table`.

**Watch for in the output:** a second table with its own independent snapshot history.

In [ ]:
processed_namespace = "co2.processed_data"
processed_table_name = f"{processed_namespace}.emissions"

# TODO: create the processed namespace and an empty table for the cleaned data.
#       Name the table object `iceberg_processed_table`.
#       Hint: build its schema from df_processed, not from df_2020.
# TODO: convert df_processed to PyArrow and append it, with snapshot properties.

# YOUR CODE HERE

## Part 4: Schema evolution (the 2021 update)

Data providers change their formats. The 2021 extract has **dropped two columns**,
and it is also missing rows for several countries — you will fix those in Part 7.

Because the new file supersedes the old one rather than extending it, the correct
operation is an overwrite, not an append.

In [ ]:
df_2021 = load_co2_csv(f"{DATA_DIR}/co2_data_until_2021_incomplete.csv")

---
### ❄️ Iceberg Operation — SCHEMA CHECK

**What you are doing:** finding out programmatically which columns the table has that the new file does not. Store the result in `dropped_columns`.

**Watch for in the output:** the two dropped column names. Do not hard-code them: a real pipeline does not know in advance what the provider changed.

In [ ]:
# TODO: compare iceberg_raw_table.schema().fields against df_2021.columns to find
#       the columns that exist in the table but not in the new data.
#       Store them in `dropped_columns`.
#       You may use compare_schemas(), or write the set arithmetic yourself.

dropped_columns =  # YOUR CODE HERE

print("Dropped columns:", sorted(dropped_columns))

---
### ❄️ Iceberg Operation — SCHEMA EVOLUTION

**What you are doing:** removing those columns from the table's schema so the new file fits.

**Watch for in the output:** the operation completing instantly. Iceberg writes a new metadata file recording the revised schema; it does not read, rewrite or delete a single Parquet file.

**Corresponds to:** Figure 6 in the paper.

In [ ]:
# TODO: inside the update_schema() context manager, delete each dropped column.

with iceberg_raw_table.update_schema() as update:
    # YOUR CODE HERE
    pass

print("Schema is now:")
for field in iceberg_raw_table.schema().fields:
    print(f"  field_id={field.field_id:<3} {field.name:<18} {field.field_type}")

Look at the `field_id` values printed above, and compare the file counts below with
what `show_warehouse()` printed in Part 2. Part 8 asks you about both.

Notice that the surviving columns kept their original field IDs. Those IDs are how
Iceberg identifies a column inside a Parquet file that was written before the schema
changed, which is what makes the time travel in Part 5 possible.

In [ ]:
show_warehouse(iceberg_raw_table, label="after dropping two columns")

---
### ❄️ Iceberg Operation — OVERWRITE

**What you are doing:** replacing the entire contents of the raw table with the 2021 data, under the revised schema.

**Watch for in the output:** **two** new snapshots rather than one — a `delete` followed by an `append`. An overwrite is a delete-and-append pair committed atomically. (Some PyIceberg versions report it as a single `overwrite` snapshot.)

**Corresponds to:** Figure 4 in the paper.

In [ ]:
# TODO: overwrite iceberg_raw_table with df_2021 (converted to PyArrow).
#       Include snapshot properties, and mention the schema change in the comment.

# YOUR CODE HERE

print(iceberg_raw_table.scan().to_pandas().shape)

## Part 5: Time travel

The table's present has a different shape from its past. Both are still readable.

In [ ]:
# TODO: list every snapshot of iceberg_raw_table, printing the ID, the operation,
#       and the comment you attached at write time.

# YOUR CODE HERE

---
### ❄️ Iceberg Operation — TIME TRAVEL

**What you are doing:** reading the raw table as it stood after the very first load, when it still had seven columns and ended at 2020.

**Watch for in the output:** the row count matching the 2020 file, and the dropped columns present again. Use `find_snapshot(table, contains=...)` rather than an index: re-running any write cell shifts every index after it, and a positional lookup would then silently return the wrong data.

**Corresponds to:** Figure 5 in the paper.

In [ ]:
# TODO: use find_snapshot() to locate the snapshot from your Part 2 append,
#       scan the table as of that snapshot, and print:
#         - the shape, to show it is the smaller 2020 dataset
#         - the column list, to show the dropped columns are back

# YOUR CODE HERE

## Part 6: Partitioning the processed data

Modern emissions are queried far more often than 1950s figures. We partition the
processed data into two eras, `pre_1990` and `post_1990`, so that a query filtered
to the modern era does not have to touch the historical files.

Note this is an *explicit* partition column, chosen for clarity. Iceberg also
supports hidden partitioning, where the partition value is derived from an existing
column by a transform such as `year(...)`.

*Data preparation step — ordinary pandas work, no Iceberg involved. Provided for you.*

Read the current processed data and add the partition column.

In [ ]:
df_for_partition = iceberg_processed_table.scan().to_pandas()

df_for_partition["era_partition"] = df_for_partition["year"].apply(
    lambda y: "pre_1990" if y < 1990 else "post_1990"
)

df_for_partition["era_partition"].value_counts()

---
### ❄️ Iceberg Operation — CREATE PARTITIONED TABLE

**What you are doing:** building a schema that includes the partition column, constructing a `PartitionSpec` for it, and creating and populating the table. Name the table object `iceberg_partitioned_table`.

**Watch for in the output:** the partition spec attached to the table, and one set of data files per partition value. Look the partition column's field ID up by name with `schema.find_field(\"era_partition\").field_id` — do not hard-code a number, or this cell breaks the next time a column changes.

**Corresponds to:** Figure 7 in the paper.

In [ ]:
partitioned_table_name = f"{processed_namespace}.emissions_partitioned"

# TODO: 1. build a schema for df_for_partition (it now has the era_partition column)
# TODO: 2. find the field_id of "era_partition" by name, not by hard-coding
# TODO: 3. build a PartitionSpec with a single PartitionField:
#            source_id = the field id you just found
#            field_id  = 1000
#            name      = "era_partition"
#            transform = "identity"
# TODO: 4. drop the table if it exists, then create it with schema + partition_spec.
#          Name the table object `iceberg_partitioned_table`.
# TODO: 5. append df_for_partition (converted to PyArrow) with snapshot properties.

# YOUR CODE HERE

---
### ❄️ Iceberg Operation — PARTITION PRUNING

**What you are doing:** planning three scans and reporting how much data each would actually read.

**Watch for in the output:** the record counts, not the elapsed time. On a dataset this small the wall clock tells you nothing; the planned record count is the mechanism.

In [ ]:
def scan_cost(scan):
    # Returns (number of data files, number of records) the scan plans to read.
    tasks = list(scan.plan_files())
    return len(tasks), sum(task.file.record_count for task in tasks)

# TODO: build these three scans and print the cost of each using scan_cost():
#   1. the partitioned table with no filter
#   2. the partitioned table filtered to era_partition == 'post_1990'
#   3. the unpartitioned processed table filtered to year >= 1990
# Then print the head of the filtered partitioned scan.

# YOUR CODE HERE

## Part 7: Atomic upsert on the raw data

Urgent notification: the 2021 rows for five countries were never reported, and have
now arrived in a separate file. You must add them to the **raw** table.

An upsert is a delete followed by an insert. Here the rows are missing rather than
wrong, so the delete will match nothing — but you should still write it. That is
what makes the operation *idempotent*: running it twice produces the same result
instead of duplicating rows. Both halves must go inside a single transaction, so a
reader can never observe a state where the delete happened and the append did not.

In [ ]:
df_updates = load_co2_csv(f"{DATA_DIR}/co2_data_until_2021_updates.csv")

df_updates

---
### ❄️ Iceberg Operation — ATOMIC TRANSACTION (delete + append)

**What you are doing:** deleting any existing 2021 rows for the affected countries and appending the new rows, as one indivisible commit.

**Watch for in the output:** a single new commit covering both operations. Build the delete expression with `And(EqualTo(...), In(...))`.

In [ ]:
countries_to_update = tuple(df_updates["country"].unique())
print("Adding 2021 rows for:", countries_to_update)

# TODO: open a transaction on iceberg_raw_table.
#   1. stage a delete of rows where year == 2021 AND country is in countries_to_update
#      (use And, EqualTo and In)
#   2. stage an append of df_updates converted to PyArrow, with snapshot properties
#   3. let the context manager commit

# YOUR CODE HERE

print("\nUpsert committed.")

---
### ❄️ Iceberg Operation — VERIFY

**What you are doing:** proving the five countries now have 2021 rows in the **raw** table.

**Watch for in the output:** five rows, one per country. Note this is the raw table, not the processed one — the processed table has not been rebuilt since the correction arrived.

In [ ]:
# TODO: scan iceberg_raw_table, filter the resulting DataFrame to year == 2021 and
#       country in countries_to_update, and display it.

# YOUR CODE HERE

> **Think about it:** the processed table is now stale — it was built before these
> five rows existed. In a real pipeline, what would trigger the processed table to be
> rebuilt, and how would Iceberg's snapshot history help you prove afterwards which
> version of the raw data each processed version was derived from?

## Part 8: Reflection

Write a brief response (3–5 sentences each) to the following. Answer in your own
words, and where a question asks for evidence, quote the actual numbers your
notebook printed.

1. **Schema evolution.** In Part 4 you dropped two columns. How does this differ from
   an `ALTER TABLE DROP COLUMN` in a traditional relational database, and why does the
   difference matter for a data lake holding terabytes?

2. **Time travel.** In Part 5 you read a state of the table that no longer exists.
   Describe a specific situation in a data team where this would be critical, and what
   the team would have had to do instead without it.

3. **Partitioning.** In Part 6 you compared three scan plans. Quote the record counts
   you obtained and explain what produced the difference. Why is counting planned
   records a better demonstration than timing the queries?

4. **The metadata layer.** Compare the `show_warehouse()` output from Part 2 with the
   one from Part 4, after the column drop. What changed, what did not, and what does
   that tell you about where a schema actually lives? Then look at the `field_id`
   values you printed after the drop: the surviving columns kept the numbers they
   already had. Suppose the dropped column had been in the *middle* of the table
   rather than at the end — what would the remaining IDs look like, and why does
   Iceberg refuse to renumber them?

### Your answers

*Replace this cell with your responses.*

**1. Schema evolution.**

**2. Time travel.**

**3. Partitioning.**

**4. The metadata layer.**